# Generate OpenICU configs from RICU

This example converts RICU `concept-dict.json` entries into OpenICU YAML configs. It is config-driven: edit `config/ricu_converter.yml` instead of editing code cells.

The example covers the RICU source keys `miiv`, `mimic_demo`, `eicu`, and `eicu_demo`.

In [ ]:
from pathlib import Path
import yaml

from open_icu.adapter.converters.ricu import build_generation_plan, write_generation_plan

NOTEBOOK_DIR = Path.cwd()
CONFIG_PATH = NOTEBOOK_DIR / "config" / "ricu_converter.yml"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def expand(path_value):
    return Path(path_value).expanduser().resolve()

paths = cfg["paths"]
conversion = cfg["conversion"]

CONCEPT_DICT = expand(paths["concept_dict"])
OPENICU_CONFIG = expand(paths["openicu_config"])
OUTPUT_CONFIG = expand(paths["output_config"])

print("concept_dict:", CONCEPT_DICT)
print("openicu_config:", OPENICU_CONFIG)
print("output_config:", OUTPUT_CONFIG)

## Build one combined generation plan

This is the normal mode if you want all supported RICU source variants in one generated OpenICU config tree.

In [ ]:
sources = conversion["sources"]

plan = build_generation_plan(
    concept_dict_path=CONCEPT_DICT,
    sources=sources,
    openicu_config=OPENICU_CONFIG,
    concepts=conversion.get("concepts"),
    categories=conversion.get("categories"),
    include_derived=conversion.get("include_derived", False),
    complex_stubs=conversion.get("complex_stubs", False),
    regex_prefix_mode=conversion.get("regex_prefix_mode"),
    logical_columns_mode=conversion.get("logical_columns_mode"),
)

print("sources:", sources)
print("generated files:", len(plan.files))
print("unsupported entries:", len(plan.unsupported))
plan.report

In [ ]:
written = write_generation_plan(
    plan,
    OUTPUT_CONFIG,
    overwrite=conversion.get("overwrite", False),
    report_path="ricu_unsupported.yml",
)

print(f"written files: {len(written)}")
for path in written[:20]:
    print(path)
if len(written) > 20:
    print("...")

## Generate each source separately

This is useful if you want to inspect which files are produced by `miiv`, `mimic_demo`, `eicu`, and `eicu_demo` individually before writing them into the same OpenICU config tree.

In [ ]:
per_source_plans = {}
for source in sources:
    source_plan = build_generation_plan(
        concept_dict_path=CONCEPT_DICT,
        sources=[source],
        openicu_config=OPENICU_CONFIG,
        concepts=conversion.get("concepts"),
        categories=conversion.get("categories"),
        include_derived=conversion.get("include_derived", False),
        complex_stubs=conversion.get("complex_stubs", False),
        regex_prefix_mode=conversion.get("regex_prefix_mode"),
        logical_columns_mode=conversion.get("logical_columns_mode"),
    )
    per_source_plans[source] = source_plan
    print(f"{source:12s} files={len(source_plan.files):5d} unsupported={len(source_plan.unsupported):5d}")

## About common vs dataset-specific configs

The converter intentionally separates global concept metadata from dataset-specific mappings:

- `concept/<category>/<concept>.yml` is shared/global.
- `dataset/<dataset>/<version>/concept/<concept>.yml` is dataset-specific.

So you should not manually copy-paste generated global configs per dataset. Generate all requested sources together, or generate per source into a temporary output directory and review the differences. The writer merges identical global files and compatible simple mapping collisions; incompatible collisions raise an error.

In [ ]:
# Optional: inspect paths that appear in more than one per-source plan.
from collections import defaultdict

path_sources = defaultdict(list)
for source, source_plan in per_source_plans.items():
    for generated in source_plan.files:
        path_sources[generated.path].append(source)

shared_paths = {path: srcs for path, srcs in path_sources.items() if len(srcs) > 1}
print("shared generated paths:", len(shared_paths))
for path, srcs in list(shared_paths.items())[:30]:
    print(path, "<-", srcs)